# RoBERTa v2 — Author Profiling (PAN14 → LiLaH)

Improvements over v1:
- `cardiffnlp/twitter-roberta-base` — pretrained on 58M tweets, closer domain than generic RoBERTa
- `MAX_LENGTH = 256` — captures ~2× more text per example (v1 truncated 70% of each chunk)
- **Focal loss** (γ=2) — focuses training on hard/minority examples, better than weighted CE for 66- imbalance
- 8 epochs + gradient accumulation (effective batch = 32)

**Runtime:** ~35 min on Colab T4  
**Steps:**
1. `Runtime → Change runtime type → T4 GPU`
2. Run **cell 1 only** → installs packages and auto-restarts the kernel
3. After restart, run **cells 2 onwards** (skip cell 1)

In [ ]:
# ── 1. Install / upgrade then restart ────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
                       "transformers", "accelerate", "scikit-learn"])
import os
os.kill(os.getpid(), 9)

In [ ]:
# ── 2. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 3. Configuration ──────────────────────────────────────────────────────────
import os

DRIVE_BASE = "/content/drive/MyDrive"   # adjust if files are in a subfolder
TRAIN_CSV  = os.path.join(DRIVE_BASE, "pan14_colab.csv")
TEST_TSV   = os.path.join(DRIVE_BASE, "hate_speech_only.tsv")

# Model — twitter-pretrained RoBERTa, much closer domain than roberta-base
MODEL_NAME = "cardiffnlp/twitter-roberta-base"

MAX_LENGTH   = 256   # v1 used 128; pan14_colab texts are ~400 tokens on average
BATCH_SIZE   = 16
GRAD_ACCUM   = 2     # effective batch = 32
LR           = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS   = 8
WARMUP_RATIO = 0.1
FOCAL_GAMMA  = 2.0   # focal loss focusing parameter; 0 = standard CE
SEED         = 42

GENDER_LABELS = ["F", "M"]
AGE_LABELS    = ["0-25", "26-35", "36-65", "66-"]

GENDER_MODEL_DIR = os.path.join(DRIVE_BASE, "roberta_v2_gender_model")
AGE_MODEL_DIR    = os.path.join(DRIVE_BASE, "roberta_v2_age_model")
os.makedirs(GENDER_MODEL_DIR, exist_ok=True)
os.makedirs(AGE_MODEL_DIR,    exist_ok=True)

print("Config OK")

In [ ]:
# ── 4. Imports and GPU check ──────────────────────────────────────────────────
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_recall_fscore_support
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, set_seed
)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU — enable T4 in Runtime settings.")

In [ ]:
# ── 5. Load data ──────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
train_df["text"]   = train_df["text"].astype(str).str.strip()
train_df["gender"] = train_df["gender"].astype(str).str.strip().str.upper()
train_df["age"]    = train_df["age"].astype(str).str.strip()
train_df = train_df.dropna(subset=["text", "gender", "age"]).reset_index(drop=True)

test_df = pd.read_csv(TEST_TSV, sep="\t")
test_df["text"]   = test_df["text"].astype(str).str.strip()
test_df["gender"] = test_df["gender"].astype(str).str.strip().str.upper()
test_df["age"]    = test_df["age"].astype(str).str.strip()
test_df = test_df.dropna(subset=["text", "gender", "age"]).reset_index(drop=True)

print(f"Train (PAN14): {len(train_df)} rows")
print(f"Test  (LiLaH): {len(test_df)} rows")
print("\nTrain gender:", train_df["gender"].value_counts().to_dict())
print("Train age:   ", train_df["age"].value_counts().to_dict())
print("\nTest gender: ", test_df["gender"].value_counts().to_dict())
print("Test age:    ", test_df["age"].value_counts().to_dict())

In [ ]:
# ── 6. Dataset class ──────────────────────────────────────────────────────────
class AuthorDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
# ── 7. Focal loss trainer and utilities ───────────────────────────────────────

class FocalLossTrainer(Trainer):
    """
    Focal loss: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    gamma=2 focuses training on hard examples (minority classes like 66-).
    alpha = class weights from compute_class_weight("balanced").
    """
    def __init__(self, class_weights, gamma=2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        alpha   = self.class_weights.to(logits.device)

        # Standard cross-entropy per sample (with class weights)
        ce = F.cross_entropy(logits, labels, weight=alpha, reduction="none")
        # Focal modulation: down-weight easy examples
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma * ce).mean()

        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1,
        "macro_precision": p,
        "macro_recall": r,
    }


def plot_confusion_matrix(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()


def build_label_maps(labels):
    label2id = {l: i for i, l in enumerate(labels)}
    id2label  = {i: l for l, i in label2id.items()}
    return label2id, id2label


print("Utilities defined.")

---
## Task 1 — Gender

In [ ]:
# ── 8. Prepare gender data ────────────────────────────────────────────────────
label2id_g, id2label_g = build_label_maps(GENDER_LABELS)

g_train = train_df[train_df["gender"].isin(GENDER_LABELS)].copy()
g_test  = test_df[test_df["gender"].isin(GENDER_LABELS)].copy()
g_train["label"] = g_train["gender"].map(label2id_g)
g_test["label"]  = g_test["gender"].map(label2id_g)

g_train_df, g_val_df = train_test_split(
    g_train, test_size=0.1, stratify=g_train["label"], random_state=SEED
)
print(f"Gender  train={len(g_train_df)}  val={len(g_val_df)}  test={len(g_test)}")

cw_g = compute_class_weight("balanced", classes=np.array(GENDER_LABELS), y=g_train["gender"].tolist())
cw_g_tensor = torch.tensor(cw_g, dtype=torch.float)
print(f"Class weights: { {GENDER_LABELS[i]: round(cw_g[i], 3) for i in range(len(GENDER_LABELS))} }")

In [ ]:
# ── 9. Tokenise ───────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

g_train_ds = AuthorDataset(g_train_df["text"].tolist(), g_train_df["label"].tolist(), tokenizer)
g_val_ds   = AuthorDataset(g_val_df["text"].tolist(),   g_val_df["label"].tolist(),   tokenizer)
g_test_ds  = AuthorDataset(g_test["text"].tolist(),     g_test["label"].tolist(),     tokenizer)
print("Tokenisation done.")

In [ ]:
# ── 10. Train gender model ────────────────────────────────────────────────────
gender_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(GENDER_LABELS), id2label=id2label_g, label2id=label2id_g,
    ignore_mismatched_sizes=True
)

gender_args = TrainingArguments(
    output_dir=GENDER_MODEL_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

gender_trainer = FocalLossTrainer(
    class_weights=cw_g_tensor,
    gamma=FOCAL_GAMMA,
    model=gender_model,
    args=gender_args,
    train_dataset=g_train_ds,
    eval_dataset=g_val_ds,
    compute_metrics=compute_metrics,
)

print("Training gender model...")
t0 = time.time()
gender_trainer.train()
print(f"Done in {(time.time()-t0)/60:.1f} min")
gender_trainer.save_model(GENDER_MODEL_DIR)
tokenizer.save_pretrained(GENDER_MODEL_DIR)
print(f"Saved to {GENDER_MODEL_DIR}")

In [ ]:
# ── 11. Evaluate gender on LiLaH ─────────────────────────────────────────────
g_out = gender_trainer.predict(g_test_ds)
g_pred_labels = [id2label_g[i] for i in np.argmax(g_out.predictions, axis=1)]
g_true_labels = [id2label_g[i] for i in g_out.label_ids]

print("\n" + "="*60)
print("GENDER — twitter-RoBERTa v2 on LiLaH")
print("="*60)
print(f"Accuracy:   {accuracy_score(g_true_labels, g_pred_labels):.4f}")
print(f"Macro F1:   {f1_score(g_true_labels, g_pred_labels, average='macro', zero_division=0):.4f}")
print(f"Weighted F1:{f1_score(g_true_labels, g_pred_labels, average='weighted', zero_division=0):.4f}")
print()
print(classification_report(g_true_labels, g_pred_labels, labels=GENDER_LABELS, zero_division=0))
plot_confusion_matrix(g_true_labels, g_pred_labels, GENDER_LABELS, "Gender — twitter-RoBERTa v2")

g_test["pred_gender"] = g_pred_labels
g_test.to_csv(os.path.join(DRIVE_BASE, "roberta_v2_gender_predictions.csv"), index=False)
print("Predictions saved.")

---
## Task 2 — Age

In [ ]:
# ── 12. Prepare age data ──────────────────────────────────────────────────────
label2id_a, id2label_a = build_label_maps(AGE_LABELS)

a_train = train_df[train_df["age"].isin(AGE_LABELS)].copy()
a_test  = test_df[test_df["age"].isin(AGE_LABELS)].copy()
a_train["label"] = a_train["age"].map(label2id_a)
a_test["label"]  = a_test["age"].map(label2id_a)

a_train_df, a_val_df = train_test_split(
    a_train, test_size=0.1, stratify=a_train["label"], random_state=SEED
)
print(f"Age  train={len(a_train_df)}  val={len(a_val_df)}  test={len(a_test)}")
print("Train dist:", a_train_df["age"].value_counts().to_dict())

# Balanced class weights then manually boost 66- further
cw_a = compute_class_weight("balanced", classes=np.array(AGE_LABELS), y=a_train["age"].tolist())
cw_a[AGE_LABELS.index("66-")] *= 1.5   # extra boost for 66-
cw_a_tensor = torch.tensor(cw_a, dtype=torch.float)
print(f"Class weights: { {AGE_LABELS[i]: round(cw_a[i], 3) for i in range(len(AGE_LABELS))} }")

In [ ]:
# ── 13. Tokenise ──────────────────────────────────────────────────────────────
a_train_ds = AuthorDataset(a_train_df["text"].tolist(), a_train_df["label"].tolist(), tokenizer)
a_val_ds   = AuthorDataset(a_val_df["text"].tolist(),   a_val_df["label"].tolist(),   tokenizer)
a_test_ds  = AuthorDataset(a_test["text"].tolist(),     a_test["label"].tolist(),     tokenizer)
print("Tokenisation done.")

In [ ]:
# ── 14. Train age model ───────────────────────────────────────────────────────
age_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(AGE_LABELS), id2label=id2label_a, label2id=label2id_a,
    ignore_mismatched_sizes=True
)

age_args = TrainingArguments(
    output_dir=AGE_MODEL_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

age_trainer = FocalLossTrainer(
    class_weights=cw_a_tensor,
    gamma=FOCAL_GAMMA,
    model=age_model,
    args=age_args,
    train_dataset=a_train_ds,
    eval_dataset=a_val_ds,
    compute_metrics=compute_metrics,
)

print("Training age model...")
t0 = time.time()
age_trainer.train()
print(f"Done in {(time.time()-t0)/60:.1f} min")
age_trainer.save_model(AGE_MODEL_DIR)
tokenizer.save_pretrained(AGE_MODEL_DIR)
print(f"Saved to {AGE_MODEL_DIR}")

In [ ]:
# ── 15. Evaluate age on LiLaH ─────────────────────────────────────────────────
a_out = age_trainer.predict(a_test_ds)
a_pred_labels = [id2label_a[i] for i in np.argmax(a_out.predictions, axis=1)]
a_true_labels = [id2label_a[i] for i in a_out.label_ids]

print("\n" + "="*60)
print("AGE — twitter-RoBERTa v2 on LiLaH")
print("="*60)
print(f"Accuracy:   {accuracy_score(a_true_labels, a_pred_labels):.4f}")
print(f"Macro F1:   {f1_score(a_true_labels, a_pred_labels, average='macro', zero_division=0):.4f}")
print(f"Weighted F1:{f1_score(a_true_labels, a_pred_labels, average='weighted', zero_division=0):.4f}")
print()
print(classification_report(a_true_labels, a_pred_labels, labels=AGE_LABELS, zero_division=0))
plot_confusion_matrix(a_true_labels, a_pred_labels, AGE_LABELS, "Age — twitter-RoBERTa v2")

a_test["pred_age"] = a_pred_labels
a_test.to_csv(os.path.join(DRIVE_BASE, "roberta_v2_age_predictions.csv"), index=False)
print("Predictions saved.")

In [ ]:
# ── 16. Summary + comparison with v1 ─────────────────────────────────────────
from sklearn.metrics import f1_score as _f1

g_f1_per = _f1(g_true_labels, g_pred_labels, labels=GENDER_LABELS, average=None, zero_division=0)
a_f1_per = _f1(a_true_labels, a_pred_labels, labels=AGE_LABELS,    average=None, zero_division=0)

print("\n" + "="*65)
print("FINAL SUMMARY — twitter-RoBERTa v2 (PAN14 → LiLaH)")
print("="*65)

print("\nGENDER")
print(f"  Accuracy:    {accuracy_score(g_true_labels, g_pred_labels):.4f}")
print(f"  Macro F1:    {_f1(g_true_labels, g_pred_labels, average='macro', zero_division=0):.4f}")
for lbl, s in zip(GENDER_LABELS, g_f1_per):
    print(f"  F1 ({lbl}):    {s:.4f}")

print("\nAGE")
print(f"  Accuracy:    {accuracy_score(a_true_labels, a_pred_labels):.4f}")
print(f"  Macro F1:    {_f1(a_true_labels, a_pred_labels, average='macro', zero_division=0):.4f}")
for lbl, s in zip(AGE_LABELS, a_f1_per):
    print(f"  F1 ({lbl}): {s:.4f}")

print("\n" + "-"*65)
print("v1 reference (roberta-base, MAX_LENGTH=128, weighted CE, 5 ep):")
print("  Gender Macro F1: 0.4381  |  Age Macro F1: 0.3006  |  66- F1: 0.2185")
print("  BERT reference:")
print("  Gender Macro F1: 0.4866  |  Age Macro F1: 0.3105  |  66- F1: 0.2937")